# Complete Databricks to Azure Machine Learning Integration (SDK v2)

This notebook demonstrates full integration between Azure Databricks and Azure Machine Learning using the AzureML SDK v2.

## Overview
This notebook covers:
1. Installing AzureML SDK v2 dependencies
2. Configuring AzureML client with authentication
3. Submitting command jobs to AzureML
4. Registering models in AzureML
5. Triggering AutoML classification jobs
6. Logging metrics to AzureML experiments
7. Creating AzureML pipelines that call Databricks jobs
8. End-to-end integration example

## Prerequisites
- Azure Databricks workspace
- Azure Machine Learning workspace
- Appropriate permissions to access both services
- Azure subscription credentials

## Important Notes
- Replace all placeholder values marked with `<...>` before running
- Store sensitive values in Azure Key Vault or Databricks secrets
- Never hardcode credentials in notebooks

---
## Section 1: Install Dependencies

Install the required Azure ML SDK v2 packages:
- `azure-ai-ml`: Azure Machine Learning SDK v2
- `azure-identity`: Azure authentication library

In [ ]:
%pip install azure-ai-ml azure-identity

---
## Section 2: Configure AzureML Client

Set up authentication and create an MLClient instance to interact with Azure Machine Learning.

### Required Configuration
Replace the following placeholders:
- `<SUBSCRIPTION_ID>`: Your Azure subscription ID
- `<RESOURCE_GROUP>`: Resource group containing your AzureML workspace
- `<AML_WORKSPACE>`: Name of your AzureML workspace

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

# Azure ML Workspace Configuration
# TODO: Replace these placeholders with your actual values
subscription_id = "<SUBSCRIPTION_ID>"
resource_group = "<RESOURCE_GROUP>"
workspace_name = "<AML_WORKSPACE>"

# Authenticate using DefaultAzureCredential
# This will use managed identity, environment variables, or Azure CLI credentials
credential = DefaultAzureCredential()

# Create MLClient
ml_client = MLClient(
    credential=credential,
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace_name
)

print(f"✓ Connected to AzureML workspace: {workspace_name}")
print(f"  Resource Group: {resource_group}")
print(f"  Subscription: {subscription_id}")

---
## Section 3: Submit a Command Job to AzureML

Submit a training job to AzureML that runs a Python script from your local `src/` directory.

### Prerequisites
- Create a `src/` directory with your training script `train.py`
- Ensure you have a compute cluster in AzureML
- Configure the environment with required dependencies

In [ ]:
from azure.ai.ml import command
from azure.ai.ml import Input, Output

# Define the command job
job = command(
    code="./src",  # Local path to source code directory
    command="python train.py --n-estimators 100 --test-size 0.2",
    environment="AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",  # Use a curated environment
    compute="<COMPUTE_CLUSTER_NAME>",  # TODO: Replace with your compute cluster name
    display_name="databricks-training-job",
    description="Training job submitted from Databricks notebook",
    experiment_name="databricks-azureml-integration"
)

# Submit the job
returned_job = ml_client.jobs.create_or_update(job)

print(f"✓ Job submitted successfully!")
print(f"  Job Name: {returned_job.name}")
print(f"  Job ID: {returned_job.id}")
print(f"  Status: {returned_job.status}")
print(f"\n  View job in Azure Portal:")
print(f"  {returned_job.studio_url}")

---
## Section 4: Register a Model in AzureML

Load a trained model from DBFS and register it in Azure Machine Learning.

### Prerequisites
- Trained model saved in DBFS (e.g., `dbfs:/mnt/models/model.pkl`)
- Model is in a supported format (pickle, joblib, etc.)

In [ ]:
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes
import os

# Path to model in DBFS
# TODO: Replace with your actual model path
dbfs_model_path = "dbfs:/mnt/models/model.pkl"

# Convert DBFS path to local filesystem path
# DBFS paths are accessible via /dbfs/ in Databricks
local_model_path = dbfs_model_path.replace("dbfs:", "/dbfs")

# Verify model exists
if os.path.exists(local_model_path):
    print(f"✓ Model found at: {local_model_path}")
else:
    print(f"⚠ Model not found at: {local_model_path}")
    print(f"  Please ensure the model file exists before proceeding")

# Create Model entity
model = Model(
    path=local_model_path,
    type=AssetTypes.CUSTOM_MODEL,  # Use CUSTOM_MODEL for pickle/joblib files
    name="databricks-trained-model",
    description="Model trained in Databricks and registered in AzureML",
    tags={"source": "databricks", "framework": "scikit-learn"}
)

# Register the model
registered_model = ml_client.models.create_or_update(model)

print(f"\n✓ Model registered successfully!")
print(f"  Model Name: {registered_model.name}")
print(f"  Model Version: {registered_model.version}")
print(f"  Model ID: {registered_model.id}")

---
## Section 5: Trigger an AzureML AutoML Job

Launch an automated machine learning (AutoML) classification job using Azure Machine Learning.

### Prerequisites
- MLTable dataset registered in AzureML or accessible via path
- Compute cluster configured in AzureML
- Dataset must have the target column for classification

In [ ]:
from azure.ai.ml import automl
from azure.ai.ml import Input
from azure.ai.ml.constants import AssetTypes

# Define the classification job
classification_job = automl.classification(
    # Training data configuration
    training_data=Input(
        type=AssetTypes.MLTABLE,
        path="<MLTABLE_PATH>"  # TODO: Replace with your MLTable path or URI
    ),
    
    # Target column for classification
    target_column_name="<TARGET_COLUMN>",  # TODO: Replace with your target column name
    
    # Primary metric to optimize
    primary_metric="accuracy",  # Options: accuracy, AUC_weighted, precision_score_weighted, etc.
    
    # Compute configuration
    compute="<COMPUTE_CLUSTER_NAME>",  # TODO: Replace with your compute cluster name
    
    # Job configuration
    experiment_name="databricks-automl-classification",
    name="automl-classification-job",
    
    # AutoML settings
    enable_onnx_compatible_models=True,
    
    # Training configuration
    n_cross_validations=5,
    training_mode="auto"
)

# Set limits for the AutoML job
classification_job.set_limits(
    timeout_minutes=60,
    trial_timeout_minutes=20,
    max_trials=10,
    max_concurrent_trials=2,
    enable_early_termination=True
)

# Submit the AutoML job
returned_automl_job = ml_client.jobs.create_or_update(classification_job)

print(f"✓ AutoML job submitted successfully!")
print(f"  Job Name: {returned_automl_job.name}")
print(f"  Job ID: {returned_automl_job.id}")
print(f"  Status: {returned_automl_job.status}")
print(f"\n  View job in Azure Portal:")
print(f"  {returned_automl_job.studio_url}")

---
## Section 6: Log Metrics to AzureML

Create an experiment run and log custom metrics to Azure Machine Learning.

This is useful for tracking model performance metrics from Databricks training jobs.

In [ ]:
import mlflow
from azure.ai.ml import MLClient
from datetime import datetime

# Configure MLflow to track with AzureML
mlflow.set_tracking_uri(ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri)

# Set experiment name
experiment_name = "databricks-metrics-logging"
mlflow.set_experiment(experiment_name)

# Start a new run
with mlflow.start_run(run_name=f"metrics-run-{datetime.now().strftime('%Y%m%d-%H%M%S')}") as run:
    # Log parameters
    mlflow.log_param("source", "databricks")
    mlflow.log_param("model_type", "classification")
    mlflow.log_param("algorithm", "random_forest")
    
    # Log metrics
    mlflow.log_metric("accuracy", 0.95)
    mlflow.log_metric("loss", 0.12)
    mlflow.log_metric("precision", 0.93)
    mlflow.log_metric("recall", 0.94)
    mlflow.log_metric("f1_score", 0.935)
    
    # Log multiple epochs
    for epoch in range(1, 11):
        mlflow.log_metric("train_loss", 1.0 / epoch, step=epoch)
        mlflow.log_metric("val_accuracy", 0.8 + (epoch * 0.015), step=epoch)
    
    # Log tags
    mlflow.set_tags({
        "environment": "databricks",
        "team": "data-science",
        "framework": "scikit-learn"
    })
    
    run_id = run.info.run_id

print(f"✓ Metrics logged successfully!")
print(f"  Experiment: {experiment_name}")
print(f"  Run ID: {run_id}")
print(f"  Run Name: {run.info.run_name}")
print(f"\n  View in AzureML Studio:")
print(f"  https://ml.azure.com/runs/{run_id}?wsid=/subscriptions/{subscription_id}/resourcegroups/{resource_group}/workspaces/{workspace_name}")

---
## Section 7: Add AzureML Pipeline Step That Calls a Databricks Job

Create an AzureML pipeline that includes a step to trigger a Databricks job.

This enables orchestrating Databricks workloads as part of an AzureML pipeline.

### Prerequisites
- Databricks workspace URL
- Databricks job ID (existing job in Databricks)
- Databricks access token (store in Key Vault or Databricks secrets)

In [ ]:
from azure.ai.ml import dsl
from azure.ai.ml.entities import PipelineJob

# Note: DatabricksJobTask is available in azure-ai-ml >= 1.8.0
# For Databricks integration, you can use the SynapseSparkComponent or custom components
# Here's an example using a command component that calls Databricks REST API

from azure.ai.ml import command, Input, Output
from azure.ai.ml.entities import Environment

# Databricks configuration placeholders
# TODO: Replace these with your actual values
databricks_workspace_url = "<DATABRICKSWORKSPACEURL>"  # e.g., "https://adb-123456.azuredatabricks.net"
databricks_job_id = "<DATABRICKSJOBID>"  # e.g., "123456"
databricks_token_secret = "<DATABRICKS_TOKEN>"  # Store in Key Vault!

# Define a component that triggers Databricks job
databricks_job_component = command(
    name="trigger-databricks-job",
    display_name="Trigger Databricks Job",
    description="Triggers a Databricks job using REST API",
    code="./src",  # Directory containing the trigger script
    command="""python trigger_databricks.py \
        --workspace-url ${{inputs.workspace_url}} \
        --job-id ${{inputs.job_id}} \
        --token ${{inputs.token}}""",
    environment="AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
    inputs={
        "workspace_url": Input(type="string", default=databricks_workspace_url),
        "job_id": Input(type="string", default=databricks_job_id),
        "token": Input(type="string", default=databricks_token_secret)
    },
    compute="<COMPUTE_CLUSTER_NAME>"  # TODO: Replace with your compute cluster
)

# Define the pipeline
@dsl.pipeline(
    name="databricks-integration-pipeline",
    description="AzureML pipeline that triggers a Databricks job",
    compute="<COMPUTE_CLUSTER_NAME>"  # TODO: Replace with your compute cluster
)
def databricks_pipeline():
    """Pipeline that integrates with Databricks."""
    # Add the Databricks job trigger step
    databricks_step = databricks_job_component()
    return {}

# Create pipeline instance
pipeline_job = databricks_pipeline()
pipeline_job.settings.default_compute = "<COMPUTE_CLUSTER_NAME>"  # TODO: Replace

# Submit the pipeline
pipeline_run = ml_client.jobs.create_or_update(
    pipeline_job,
    experiment_name="databricks-pipeline-integration"
)

print(f"✓ Pipeline submitted successfully!")
print(f"  Pipeline Name: {pipeline_run.name}")
print(f"  Pipeline ID: {pipeline_run.id}")
print(f"  Status: {pipeline_run.status}")
print(f"\n  View pipeline in Azure Portal:")
print(f"  {pipeline_run.studio_url}")

---
## Section 8: Complete Integration Example

This section demonstrates a complete end-to-end workflow:
1. Train a simple model in Databricks
2. Register it in AzureML
3. Log metrics
4. Submit a validation job

This serves as a template for building production ML workflows.

In [ ]:
import mlflow
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
import joblib
import os
from datetime import datetime

# Configure MLflow
mlflow.set_tracking_uri(ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri)
mlflow.set_experiment("complete-databricks-azureml-integration")

print("=" * 80)
print("COMPLETE DATABRICKS + AZUREML INTEGRATION EXAMPLE")
print("=" * 80)

# Step 1: Train a model in Databricks
print("\n[1/5] Training model in Databricks...")
with mlflow.start_run(run_name=f"complete-integration-{datetime.now().strftime('%Y%m%d-%H%M%S')}") as run:
    # Load sample data
    iris = load_iris()
    X_train, X_test, y_train, y_test = train_test_split(
        iris.data, iris.target, test_size=0.2, random_state=42
    )
    
    # Train model
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    # Evaluate model
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    
    print(f"   ✓ Model trained successfully")
    print(f"     Accuracy: {accuracy:.4f}")
    print(f"     Precision: {precision:.4f}")
    print(f"     Recall: {recall:.4f}")
    
    # Step 2: Log metrics to AzureML
    print("\n[2/5] Logging metrics to AzureML...")
    mlflow.log_params({
        "n_estimators": 100,
        "random_state": 42,
        "test_size": 0.2
    })
    mlflow.log_metrics({
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall
    })
    print(f"   ✓ Metrics logged to AzureML")
    
    # Step 3: Save model locally
    print("\n[3/5] Saving model...")
    model_dir = "/tmp/databricks_model"
    os.makedirs(model_dir, exist_ok=True)
    model_path = os.path.join(model_dir, "model.pkl")
    joblib.dump(model, model_path)
    print(f"   ✓ Model saved to: {model_path}")
    
    # Step 4: Register model in AzureML
    print("\n[4/5] Registering model in AzureML...")
    from azure.ai.ml.entities import Model
    from azure.ai.ml.constants import AssetTypes
    
    model_entity = Model(
        path=model_path,
        type=AssetTypes.CUSTOM_MODEL,
        name="iris-classifier-databricks",
        description=f"Iris classifier trained in Databricks with {accuracy:.4f} accuracy",
        tags={
            "source": "databricks",
            "framework": "scikit-learn",
            "algorithm": "random_forest",
            "accuracy": f"{accuracy:.4f}"
        }
    )
    
    registered_model = ml_client.models.create_or_update(model_entity)
    print(f"   ✓ Model registered in AzureML")
    print(f"     Model Name: {registered_model.name}")
    print(f"     Version: {registered_model.version}")
    
    # Step 5: Log model to MLflow
    print("\n[5/5] Logging model to MLflow...")
    mlflow.sklearn.log_model(model, "model")
    print(f"   ✓ Model logged to MLflow")
    
    run_id = run.info.run_id

print("\n" + "=" * 80)
print("✓ INTEGRATION COMPLETE!")
print("=" * 80)
print(f"\nRun Details:")
print(f"  Run ID: {run_id}")
print(f"  Model: {registered_model.name} (v{registered_model.version})")
print(f"  Accuracy: {accuracy:.4f}")
print(f"\nView in AzureML Studio:")
print(f"  https://ml.azure.com/runs/{run_id}?wsid=/subscriptions/{subscription_id}/resourcegroups/{resource_group}/workspaces/{workspace_name}")
print("\n" + "=" * 80)

---
## Summary

This notebook demonstrated:

✅ **Installed** Azure ML SDK v2 and authentication libraries  
✅ **Configured** MLClient with DefaultAzureCredential  
✅ **Submitted** command jobs to AzureML compute  
✅ **Registered** models from Databricks to AzureML  
✅ **Triggered** AutoML classification jobs  
✅ **Logged** metrics and parameters to AzureML experiments  
✅ **Created** pipelines that integrate with Databricks jobs  
✅ **Built** end-to-end integration examples  

### Next Steps

1. **Replace all placeholders** with your actual Azure and Databricks values
2. **Store secrets securely** in Azure Key Vault or Databricks secrets
3. **Create the required resources**:
   - Compute clusters in AzureML
   - Training scripts in `src/` directory
   - MLTable datasets for AutoML
4. **Test each section** independently before running the complete workflow
5. **Monitor jobs** in AzureML Studio for execution status

### Additional Resources

- [Azure ML SDK v2 Documentation](https://learn.microsoft.com/azure/machine-learning/)
- [Azure Databricks Documentation](https://learn.microsoft.com/azure/databricks/)
- [MLflow on Azure ML](https://learn.microsoft.com/azure/machine-learning/how-to-use-mlflow)
- [AutoML in Azure ML](https://learn.microsoft.com/azure/machine-learning/concept-automated-ml)